## Daten laden


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

df = pd.read_parquet('../data/processed/credit_dtypes_pcd.parquet')
df.dtypes

---
## Merkmale trennen und Kategorien in Zahlen umwandeln

In [ ]:
X = df.drop(columns='loan_status')
y = df['loan_status'].astype(int)

X['cb_person_default_on_file'] = X['cb_person_default_on_file'].astype(float)
X['kredit_einkommen'] = X['loan_amnt'] / X['person_income']
X = pd.get_dummies(X, columns=['person_home_ownership', 'loan_intent'], drop_first=True, dtype=float)

X.head()

---
## Train-Test Split 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

---
## Trainings Mediane abgrenzen
Sonst würde ein Data-leakage ein zu gutes Ergebnis aka overfitting erzeugen
Eigentlich gehört dieser Schritt in 02_Datapreparation aber da es einen Mergeconflict mit Carl geben könnte, ist es besser hier aufgehoben.

Der Gedanke hinter dem Median: Die Daten zu löschen würde uns 10% des Datensatzes kosten. Wenn wir den Median in die N.A werte einfüllen, dann gibt es keine Ausreißer und wir können die Daten trotzdem noch verwenden.

Man könnte auch noch ein Schätzungsmodell für die Daten bauen aber das ist für die paar daten zu aufwendig..

In [ ]:
mediane = X_train.median()
X_train = X_train.fillna(mediane)
X_test = X_test.fillna(mediane)

X_train.isna().sum().sum(), X_test.isna().sum().sum()   # sollte (0, 0) ergeben

## Beweis: Mit gelöschten 10% der Daten würde es ca so aussehen:
### Ein paar Dinge werden jetzt vorweggegriffen
Das interessante ist. Das Modell mit den aufgefüllten Medianen outperformt das Modell mit den gelöschten Spalten sogar auf dem Datensatz mit den gelöschten Spalten.

In [ ]:
# Neu splitten, raw bedeutet: noch nicht behandelt, noch mit NaNs
X_train_raw, X_test_raw, y_train, y_test = train_test_split( 
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Variante A: fehlende Werte mit Median füllen, so wie oben bzw eig in Data Preparation
mediane = X_train_raw.median()
model_a = LinearRegression().fit(X_train_raw.fillna(mediane), y_train)

# Variante B: Zeilen mit NaNs löschen (in einer Kopie)
maske_train = X_train_raw.notna().all(axis=1) #erstellt filter für Trainingsdaten, die keine NaNs enthalten
X_train_drop = X_train_raw[maske_train].copy()
y_train_drop = y_train[maske_train].copy()
model_b = LinearRegression().fit(X_train_drop, y_train_drop) #training an den Zeilen ohne NaNs

#erstellt filter für Testdaten, die keine NaNs enthalten
maske_test = X_test_raw.notna().all(axis=1) 
print(f'Training: {(~maske_train).sum()} von {len(X_train_raw)} Zeilen gelöscht')
print(f'Test: {(~maske_test).sum()} Personen, für die Modell B keine Vorhersage machen kann\n')

# Fairer Vergleich: beide Modelle auf den vollständigen Testzeilen, rausgefiltert mit maske_test
X_test_komplett = X_test_raw[maske_test]
y_test_komplett = y_test[maske_test]

bewerten('A Median-Modell   | angewendet auf: vollständige Testzeilen mit NaNs', y_test_komplett,
         (model_a.predict(X_test_komplett) >= 0.5).astype(int))
bewerten('B NaNs-Gelöscht-Modell | angewendet auf: vollständige Testzeilen mit NaNs', y_test_komplett,
         (model_b.predict(X_test_komplett) >= 0.5).astype(int))

# Zusatz: Modell A kann auch die unvollständigen Zeilen bewerten
bewerten('A Median-Modell   |  angewendet auf: Testzeilen bereinigt von NaNs', y_test,
         (model_a.predict(X_test_raw.fillna(mediane)) >= 0.5).astype(int))
bewerten('B NaNs-Gelöscht-Modell | Testzeilen bereinigt von NaNs', y_test,
         (model_b.predict(X_test_raw.fillna(mediane)) >= 0.5).astype(int))

---
## Daten speichern

In [ ]:
from pathlib import Path

ordner = Path('../data/processed')
ordner.mkdir(parents=True, exist_ok=True)

X_train_raw.to_parquet(ordner / 'X_train_raw.parquet')
X_test_raw.to_parquet(ordner / 'X_test_raw.parquet')
y_train.to_frame().to_parquet(ordner / 'y_train.parquet')
y_test.to_frame().to_parquet(ordner / 'y_test.parquet')

---
## Bewertungsfunktion für die Ermöglichung beim Ausprobieren von Werten

In [ ]:
def bewerten(name, y_true, y_pred):
    fehler = 1 - accuracy_score(y_true, y_pred)               # Fehlerquote der tatsächlichen Vorhersagen
    erkannt = recall_score(y_true, y_pred)                  # Anteil der tatsächlich erkannten Ausfälle
    print(f'{name}| Fehlerquote {fehler:.1%} | erkannte Ausfälle {erkannt:.1%}')
    display(pd.DataFrame(confusion_matrix(y_true, y_pred),
                       index=['true: zahlt', 'true: Ausfall'],
                       columns=['pred.: zahlt', 'pred.: Ausfall']))
    print()

---
## Einfach Formel mit Beispielen einsetzen 
*!! Kein Modell verwendet, simple eigene Formel. Wirklich nur Proof of Concept*

### 1. Annahme: „Alle zahlen zurück“ also das dümmste mögliche Modell, es gibt keine Ausfälle. 
Das ist die absolute Untergrenze für die Fehlerquote und die absolute Obergrenze für erkannte Ausfälle.

In [ ]:
bewerten('Alle zahlen', y_test, [0] * len(y_test))

### 2. Annahme: Kredit über 30 % des Einkommens bedeutet es wird zu einem Ausfall kommen
Die 30% sind ein willkürlicher Wert, der in der Praxis oft als Faustregel verwendet wird.

In [ ]:

bewerten('Faustregel 30 %', y_test, (X_test['kredit_einkommen'] > 0.3).astype(int)) 

---
## Modellvergleiche 


### 1. Lineare Regression

In [ ]:
model = LinearRegression()

model.fit(X_train, y_train)          # model ist jetzt trainiert, d.h. die Koeffizienten sind bestimmt anhand bekannter Daten (X_train, y_train)

y_score = model.predict(X_test)      # Formel wird an neuen noch nie gesehenen Daten (X_test) angewendet, um Vorhersagen zu machen. y_score ist ein Array mit Werten zwischen 0 und 1, die die Wahrscheinlichkeit eines Zahlungsausfalls darstellen.

y_pred = (y_score >= 0.5).astype(int) # Schwellenwert 0.5: alles >= 0.5 wird als Ausfall (1) klassifiziert, alles < 0.5 als kein Ausfall (0)

bewerten('Lineare Regression', y_test, y_pred)

### Fazit
Man sieht, dass die Lineare Regression kaum besser ist als der Wild Guess mit den 30% von oben hahaha

Die lineare Regression kommt auf eine Fehlerquote von 15.4 % und erkennt nur 39.5 % der Ausfälle. Als Einstieg ok, aber da sie nur gerade Zusammenhänge kann und Werte außerhalb von 0 und 1 ausspuckt, ist sie für eine Ja/Nein-Frage eigentlich nicht gemacht.


---
### 2. Logistic Regression

In [ ]:
model_log = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=1000))
model_log.fit(X_train_raw, y_train)
bewerten('Logistische Regression', y_test, model_log.predict(X_test_raw))

### Fazit
Es wird schon besser
Mit 14.5 % Fehlerquote etwas besser als die lineare Regression, dafür gibt sie jetzt echte Wahrscheinlichkeiten aus. Gut erklärbar und deshalb bei Banken beliebt, aber Schwellen wie beim Kredit/Einkommen-Verhältnis kann sie nicht abbilden.

---
### 3. Decision Tree

In [ ]:
model_tree = make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=6, random_state=42))
model_tree.fit(X_train_raw, y_train)
bewerten('Entscheidungsbaum', y_test, model_tree.predict(X_test_raw))

### Fazit
Der Baum schafft 9.7 % Fehlerquote und zeigt damit, dass in den Daten Schwellen stecken, die die Regressionen nicht sehen. Man kann jede Entscheidung als Regel nachlesen, ohne die Tiefe zu begrenzen lernt er aber einfach auswendig.

---
### 4. Random Forest

In [ ]:
model_rf = make_pipeline(SimpleImputer(strategy='median'), RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42))
model_rf.fit(X_train_raw, y_train)
bewerten('Random Forest', y_test, model_rf.predict(X_test_raw))

### Fazit
Durch die 300 Bäume deutlich stabiler als ein einzelner Baum, Fehlerquote 7.6 %. Dafür kann man nicht mehr nachvollziehen, warum eine einzelne Person abgelehnt wird.

---
### 5. Gradient Boosting

In [ ]:
model_gb = HistGradientBoostingClassifier(random_state=42)
model_gb.fit(X_train_raw, y_train)
bewerten('Gradient Boosting', y_test, model_gb.predict(X_test_raw))

### Fazit
Unser bestes Modell mit 7.5 % Fehlerquote, auch weil es fehlende Werte selbst verarbeitet statt sie aufzufüllen. Die 7 % schafft es ohne Tuning aber noch nicht. Wir wenden uns in 05_Modell_Tuning mit dem Tuning beschäftigen

---
### 6. K-Nearest-Neighbor


In [ ]:
model_knn = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), KNeighborsClassifier(n_neighbors=15))
model_knn.fit(X_train_raw, y_train)
bewerten('kNN', y_test, model_knn.predict(X_test_raw))

### Fazit
kNN liegt mit 12 % deutlich hinter den Baum-Modellen, weil alle Merkmale gleich viel zählen, auch unwichtige wie das Alter. Außerdem drücken die Ausreißer beim Einkommen die Abstände zusammen.

---
### 7. Kleines Neural Network

In [ ]:
model_mlp = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=500, early_stopping=True, random_state=42)
)
model_mlp.fit(X_train_raw, y_train)
bewerten('Neuronales Netz', y_test, model_mlp.predict(X_test_raw))

### Fazit
Das Netz kommt auf 10.6 % und ist damit nicht besser als Gradient Boosting, obwohl es viel komplexer ist. Bei Tabellendaten ist das normal, Baum-Modelle sind da meistens im Vorteil.